# Binance Data Fetch (No API) + Configurable Lookback

✅ **What this notebook fixes**
- Fetches Binance OHLCV **without API credentials** (public data)
- Lets you choose **how much data to pull per timeframe** from config
- Drop-in replacement logic for your existing bot


In [ ]:
# !pip install ccxt pandas numpy
import ccxt
import pandas as pd
import numpy as np
from datetime import datetime


## ⚙️ Configuration

In [ ]:
# ── Exchange ─────────────────────────────
BROKER = 'binance'
SYMBOL = 'BTC/USDT'

# ── Timeframes ───────────────────────────
LTF = '5m'
HTF = '15m'

# ── Data Pull Control ────────────────────
# 'bars' = exact candle count
# 'days' = approximate historical range
DATA_PULL_MODE = 'bars'  # 'bars' | 'days'

LTF_BARS = 1500
HTF_BARS = 1500

BACKTEST_DAYS = 180

print('Config loaded')


## 📥 Binance OHLCV Fetch (No API Required)

In [ ]:
def binance_client():
    return ccxt.binance({
        'enableRateLimit': True,
        'options': {'adjustForTimeDifference': True}
    })

def fetch_binance_ohlcv(symbol, timeframe, bars=None, days=None):
    ex = binance_client()
    tf_minutes = {'1m':1,'5m':5,'15m':15,'30m':30,'1h':60,'4h':240,'1d':1440}

    if bars is None:
        mins = tf_minutes.get(timeframe, 60)
        bars = max(500, (days or 30) * 1440 // mins)

    all_rows = []
    since = None

    while len(all_rows) < bars:
        limit = min(1000, bars - len(all_rows))
        batch = ex.fetch_ohlcv(symbol, timeframe, since=since, limit=limit)
        if not batch:
            break
        all_rows.extend(batch)
        since = batch[-1][0] + 1

    df = pd.DataFrame(all_rows, columns=['timestamp','open','high','low','close','volume'])
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    return df.set_index('timestamp')


## 🔄 Unified Data Loader

In [ ]:
def get_data(symbol, timeframe):
    if DATA_PULL_MODE == 'bars':
        bars = LTF_BARS if timeframe == LTF else HTF_BARS
        df = fetch_binance_ohlcv(symbol, timeframe, bars=bars)
    else:
        df = fetch_binance_ohlcv(symbol, timeframe, days=BACKTEST_DAYS)

    print(f'{symbol} {timeframe} → {len(df)} candles | {df.index[0].date()} → {df.index[-1].date()}')
    return df


## ✅ Example Pull

In [ ]:
ltf = get_data(SYMBOL, LTF)
htf = get_data(SYMBOL, HTF)
ltf.tail()


## ✅ Integration Notes
Replace your existing `fetch_binance()` and `get_data()` with the versions above.

Key changes vs your original code:
- **No API key checks for Binance** (public data)
- Lookback is driven entirely by config (`LTF_BARS`, `HTF_BARS`, `BACKTEST_DAYS`)
- Works in backtest, paper, and analysis modes

If you want, I can:
- Patch this directly into your full bot file
- Add caching to avoid repeated downloads
- Add futures support (USDT-M / COIN-M)
